In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# just use and re-split test data for quicker computations to test validation pipeline
test_data = pd.read_csv("test_transaction.csv")

X_test = test_data.drop("isFraud", axis=1)
y_test = test_data["isFraud"]

# re-splitting data
X_shell_train, X_shell_val, y_shell_train, y_shell_val = train_test_split(
    X_test,
    y_test,
    test_size=0.20,
    random_state=42,
    stratify=y_test
)

In [19]:
# shell model
# logistic regression

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lg_shell_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

lg_shell_model.fit(
    X_shell_train,
    y_shell_train
)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('logistic_regression',
                 LogisticRegression(max_iter=2000, random_state=42))])

In [5]:
# shell model
# tests if validation pipeline works at all/ is valid
from sklearn.dummy import DummyClassifier

dummy_model = DummyClassifier(
    strategy="prior",
    random_state=42
)

dummy_model.fit(X_shell_train, y_shell_train)

DummyClassifier(random_state=42)

In [13]:
# validation pipeline function
# using (almost) all metrics for now
# will only use f1, avg_precision, matrix, and report in final run

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    average_precision_score
)


def validate_model(model, X_test, y_test, model_name):
    """Validate a single classification model on the test set."""

    # Generate predictions
    y_pred = model.predict(X_test)

    # Generate probabilities if supported
    y_prob = model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)
    average_precision = average_precision_score(y_test, y_prob)

    # Results for this model
    results = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "avg_precision": average_precision
    }

    print(f"\n{'=' * 50}")
    print(f"{model_name} Validation")
    print(f"{'=' * 50}")

    for metric, value in results.items():
        print(f"{metric}: {value:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    return results

In [21]:
# run pipeline on LR shell model
lg_results = validate_model(
    lg_shell_model,
    X_shell_val,
    y_shell_val,
    "Logistic Regression Shell"
)



Logistic Regression Shell Validation
accuracy: 0.9732
precision: 0.8258
recall: 0.2627
f1: 0.3985
roc_auc: 0.8541
avg_precision: 0.4413

Confusion Matrix:
[[11830    23]
 [  306   109]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99     11853
           1       0.83      0.26      0.40       415

    accuracy                           0.97     12268
   macro avg       0.90      0.63      0.69     12268
weighted avg       0.97      0.97      0.97     12268



In [15]:
# run pipeline on dummy shell
dummy_results = validate_model(
    dummy_model,
    X_shell_val,
    y_shell_val,
    "Dummy Shell"
)


Dummy Shell Validation
accuracy: 0.9662
precision: 0.0000
recall: 0.0000
f1: 0.0000
roc_auc: 0.5000
avg_precision: 0.0338

Confusion Matrix:
[[11853     0]
 [  415     0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     11853
           1       0.00      0.00      0.00       415

    accuracy                           0.97     12268
   macro avg       0.48      0.50      0.49     12268
weighted avg       0.93      0.97      0.95     12268



# Insights

## Logistic Regression
- precision, roc-auc drastically better than dummy's, as we would expect (~80% to dummy's ~50%)
- average precision low performing at 50% but still better than dummy
- recall at 20%

## Dummy 
- predicts not fraud every time
- clearly an overly simplified model
- since fraud is rare ~3% not bad
- metrics reflect the performance we'd expect (precision and recall of 0)

## Overall 
- pipeline accurately describes metrics as we'd expect (overall badly for the significant metrics)
- but, LR significantly better than dummy as expected as well
- should function well for week 9

In [ ]:
# TO BE RAN IN WEEK 9!!
# Load models...

# Validate XGBoost
xgb_results = validate_model(
    xgb_model,
    X_test,
    y_test,
    "XGBoost"
)


# Validate Random Forest
rf_results = validate_model(
    rf_model,
    X_test,
    y_test,
    "Random Forest"
)